# 🤖 Txoko RAG Chatbot — Guía de implementación
### Bootcamp BBK The Bridge · Equipo 4
**Autores:** Naia, Andoni, Unai, Fátima  
**Fecha:** Junio 2026

---

## ¿Qué es este notebook?

Este notebook documenta paso a paso cómo hemos añadido un **chatbot de lenguaje natural** a la app Txoko, una aplicación de recomendaciones turísticas para Euskadi.

El chatbot permite consultas como:
- *"Recomiéndame un restaurante de pintxos cerca del Guggenheim"*
- *"¿Qué playas tranquilas hay en Gipuzkoa?"*
- *"Busco alojamiento rural en el campo"*

---

## Índice

1. [¿Qué es RAG y por qué lo usamos?](#1-rag)
2. [Arquitectura del sistema](#2-arquitectura)
3. [Instalación de dependencias](#3-dependencias)
4. [Paso 1 — Generar embeddings y construir el índice FAISS](#4-embeddings)
5. [Paso 2 — Verificar que el índice funciona](#5-verificacion)
6. [Próximos pasos](#6-proximos)


## 1. ¿Qué es RAG y por qué lo usamos? <a name="1-rag"></a>

### El problema

Queremos un chatbot que responda preguntas sobre lugares turísticos de Euskadi usando **nuestra propia base de datos**.

Las dos opciones que podríamos considerar:

| Opción | Descripción | ¿Por qué no? |
|--------|-------------|--------------|
| **Entrenar un modelo desde cero** | Crear nuestro propio LLM con nuestros datos | Requiere millones de ejemplos, semanas de cómputo y cientos de miles de euros en GPUs |
| **Fine-tuning** | Ajustar un modelo existente con nuestros datos | Sigue requiriendo muchos datos etiquetados y es costoso. El modelo "memoriza" datos estáticos y se queda desactualizado |
| **RAG ✅** | Usar un LLM existente + buscar en nuestra BD en tiempo real | Económico, actualizable, no requiere entrenamiento |

### ¿Qué significa RAG?

**RAG = Retrieval-Augmented Generation** (Generación Aumentada por Recuperación)

La idea es sencilla: en vez de pedirle al LLM que "sepa" cosas de Euskadi, le proporcionamos la información relevante en cada consulta.

```
Usuario pregunta: "pintxos cerca del Guggenheim"
        ↓
[1] Nuestro código busca en la BD los lugares más relevantes
        ↓
[2] Nuestro código construye un prompt con esos lugares
        ↓
[3] El LLM genera una respuesta en lenguaje natural
        ↓
Usuario recibe: "Te recomiendo el Bar Txiriboga, a 200m del Guggenheim..."
```

**Clave conceptual:** El LLM no consulta nuestra base de datos directamente. Es nuestro código el que orquesta todo. El LLM solo recibe texto y genera texto.

### ¿Por qué no simplemente buscar en la BD con SQL?

Una búsqueda SQL normal (`WHERE nombre LIKE '%pintxos%'`) solo encuentra coincidencias exactas de texto.

RAG usa **búsqueda semántica**: entiende que "comida vasca tradicional", "pintxos" y "gastronomía del País Vasco" son conceptos relacionados, aunque no compartan palabras.


## 2. Arquitectura del sistema <a name="2-arquitectura"></a>

El sistema tiene dos fases bien diferenciadas:

### Fase offline (se ejecuta una vez)

```
aupa_master_v5.csv  ──┐
                       ├──► build_faiss_index.py ──► faiss_index.index
txoko_reviews_raw.json ┘                          └──► faiss_metadata.json
```

Convertimos todos nuestros lugares a **embeddings** (vectores numéricos) y los almacenamos en un índice FAISS. Esto es lo que hace este notebook.

### Fase online (se ejecuta en cada consulta)

```
Pregunta del usuario
        ↓
Convertir pregunta a embedding (mismo modelo)
        ↓
Buscar los k lugares más similares en FAISS
        ↓
Construir prompt con los lugares recuperados
        ↓
Llamar a la API de Gemini
        ↓
Devolver respuesta al usuario
```

### ¿Qué es un embedding?

Un embedding es una representación numérica de un texto como un vector de números.

```
"restaurante de pintxos en Bilbao" → [0.23, -0.45, 0.12, 0.87, ...]  (384 números)
"bar de pintxos bilbaíno"          → [0.21, -0.43, 0.14, 0.85, ...]  (muy similar)
"playa en Zarautz"                 → [0.67,  0.12, -0.34, 0.23, ...]  (muy diferente)
```

Textos con significado similar tienen vectores similares (cercanos en el espacio vectorial). Esto es lo que permite la búsqueda semántica.

### ¿Qué es FAISS?

FAISS (Facebook AI Similarity Search) es una librería que permite buscar eficientemente los vectores más similares a uno dado entre millones de vectores. En nuestro caso con ~4.600 lugares es más que suficiente con su configuración más simple.

El índice FAISS es simplemente **un fichero en disco** que se carga en memoria al arrancar el servidor. No es una base de datos con servidor propio.

### Componentes y tecnologías

| Componente | Tecnología | Por qué |
|------------|-----------|---------|
| Modelo de embeddings | `sentence-transformers` | Open source, multilingüe, corre en CPU |
| Índice vectorial | FAISS | Simple, sin servidor, fichero en disco |
| LLM | Gemini API (Google) | Tier gratuito generoso, buena calidad |
| Backend | FastAPI | Ligero, fácil de desplegar en Render |
| Despliegue | Render | Ya lo usamos para el resto del proyecto |


## 3. Instalación de dependencias <a name="3-dependencias"></a>

### Requisitos previos

- Python 3.9 o superior
- Los ficheros `aupa_master_v5.csv` y `txoko_reviews_raw.json` en la misma carpeta
- Una API key de Gemini (gratuita en [aistudio.google.com](https://aistudio.google.com))

### ⚠️ Importante: gestión segura de la API key

**Nunca escribas tu API key directamente en el código ni en un chat.**

Crea un fichero `.env` en la carpeta del proyecto:
```
GEMINI_API_KEY=tu_clave_aqui
```

Y añade `.env` a tu `.gitignore` para que nunca llegue al repositorio:
```
echo ".env" >> .gitignore
```


In [ ]:
# Instalar todas las dependencias necesarias
# Ejecutar esta celda solo una vez

# Si trabajas en local con entorno virtual:
# python -m venv venv
# source venv/bin/activate  (Mac/Linux)
# venv\Scripts\activate    (Windows PowerShell)

!pip install sentence-transformers faiss-cpu pandas numpy python-dotenv google-generativeai fastapi uvicorn
#pip install sentence-transformers faiss-cpu pandas numpy
#pip install fastapi uvicorn python-dotenv google-generativeai
# pip uninstall google-generativeai
# pip install google-genai
# pip install groq
print("✓ Dependencias instaladas")


## 4. Paso 1 — Generar embeddings y construir el índice FAISS <a name="4-embeddings"></a>

### ¿Qué hace este paso?

1. Carga el CSV con los lugares y el JSON con las reseñas
2. Para cada lugar, construye un texto combinando sus campos + sus reseñas
3. Convierte cada texto en un embedding (vector de 384 números)
4. Almacena todos los embeddings en un índice FAISS
5. Guarda el índice y los metadatos en disco

### ¿Por qué combinamos reseñas con la descripción?

La descripción oficial de un lugar suele ser genérica:
> *"Restaurante de cocina vasca tradicional"*

Una reseña real añade vocabulario natural:
> *"Las croquetas son increíbles, perfecto para una cena tranquila en familia"*

El segundo texto responde mucho mejor a consultas como "cena tranquila" o "comida casera".

### ¿Por qué filtramos reseñas con rating < 3?

Para evitar que vocabulario negativo ("sucio", "lento", "caro") quede asociado semánticamente al lugar y aparezca en búsquedas que no corresponden.

### El modelo de embeddings

Usamos `paraphrase-multilingual-MiniLM-L12-v2`:
- **Multilingüe**: funciona en español, euskera e inglés
- **Ligero**: 384 dimensiones, corre en CPU sin problema  
- **Primera ejecución**: descarga ~120MB del modelo (se cachea localmente)


In [ ]:
import json
import os
import time

import faiss
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

# ─── Configuración ────────────────────────────────────────────────────────────

CSV_PATH      = "aupa_master_v5.csv"
REVIEWS_PATH  = "txoko_reviews_raw.json"
OUTPUT_DIR    = "./rag_assets"

MODEL_NAME    = "paraphrase-multilingual-MiniLM-L12-v2"
MAX_REVIEWS   = 5   # máximo de reseñas por lugar que se incorporan al texto

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("✓ Configuración lista")


In [ ]:
# ─── Funciones de construcción de texto ──────────────────────────────────────

def load_reviews(reviews_path: str) -> dict:
    """
    Carga el JSON de reseñas y lo transforma en un dict:
        {id_lugar: [{"rating": 5, "text": "..."}, ...]}
    """
    with open(reviews_path, encoding="utf-8") as f:
        raw = json.load(f)

    reviews_by_id = {}
    for place_id, data in raw.items():
        if isinstance(data, dict) and "reviews" in data:
            reviews_by_id[place_id] = data["reviews"]
        elif isinstance(data, list):
            reviews_by_id[place_id] = data

    print(f"  Reseñas cargadas: {len(reviews_by_id):,} lugares con reseñas")
    return reviews_by_id


def build_place_text(row: pd.Series, reviews_by_id: dict) -> str:
    """
    Construye el texto que se convertirá en embedding para un lugar.
    
    Combina campos estructurados del CSV con reseñas reales de Google.
    Este texto es lo que el modelo 'lee' para entender de qué trata cada lugar.
    """
    parts = []

    parts.append(f"Nombre: {row.get('nombre', '')}")
    parts.append(f"Tipo: {row.get('subcategoria', '')} | {row.get('categoria', '')}")
    parts.append(f"Ubicación: {row.get('municipio', '')}, {row.get('territorio', '')}")

    desc = str(row.get("descripcion", "")).strip()
    if desc and desc != "nan":
        parts.append(f"Descripción: {desc}")

    # Incorporar reseñas (solo las positivas para evitar ruido semántico negativo)
    place_id = str(row.get("id", ""))
    if place_id in reviews_by_id:
        texts = [
            r["text"].strip()
            for r in reviews_by_id[place_id][:MAX_REVIEWS]
            if r.get("text", "").strip() and r.get("rating", 0) >= 3
        ]
        if texts:
            parts.append("Reseñas: " + " | ".join(texts))

    return "\n".join(parts)


def build_metadata(row: pd.Series) -> dict:
    """
    Extrae los metadatos que se guardan junto al índice.
    Son los datos que el backend devuelve al frontend con cada recomendación.
    """
    def safe(val):
        return None if pd.isna(val) else val

    return {
        "id":                 str(row.get("id", "")),
        "nombre":             str(row.get("nombre", "")),
        "categoria":          str(row.get("categoria", "")),
        "subcategoria":       str(row.get("subcategoria", "")),
        "municipio":          str(row.get("municipio", "")),
        "territorio":         str(row.get("territorio", "")),
        "lat":                safe(row.get("lat")),
        "lon":                safe(row.get("lon")),
        "descripcion":        str(row.get("descripcion", "")),
        "google_rating":      safe(row.get("google_rating")),
        "google_num_reviews": safe(row.get("google_num_reviews")),
        "web":                str(row.get("web", "")),
        "ficha_turismo":      str(row.get("ficha_turismo", "")),
    }

print("✓ Funciones definidas")


In [ ]:
# ─── 1. Cargar datos ──────────────────────────────────────────────────────────

print("[1/5] Cargando datos...")
df = pd.read_csv(CSV_PATH)
print(f"  CSV cargado: {len(df):,} lugares × {len(df.columns)} columnas")

# Filtrar lugares sin nombre (no son útiles para el chatbot)
df = df[df["nombre"].notna() & df["nombre"].str.strip().ne("")]
print(f"  Tras filtro nombre: {len(df):,} lugares")

reviews_by_id = load_reviews(REVIEWS_PATH)


In [ ]:
# ─── 2. Construir textos ──────────────────────────────────────────────────────

print("\n[2/5] Construyendo textos para embeddings...")
texts    = []
metadata = []

for _, row in df.iterrows():
    texts.append(build_place_text(row, reviews_by_id))
    metadata.append(build_metadata(row))

# Mostrar ejemplo del primer lugar para verificar que el texto tiene buena pinta
print("\n  Ejemplo — texto para embedding del primer lugar:")
print("  " + "\n  ".join(texts[0].split("\n")))

n_con_reviews = sum(1 for m in metadata if m["id"] in reviews_by_id)
print(f"\n  Lugares con reseñas incorporadas: {n_con_reviews:,} / {len(texts):,}")


In [ ]:
# ─── 3. Generar embeddings ────────────────────────────────────────────────────
#
# Este paso puede tardar entre 3 y 10 minutos en CPU.
# El modelo se descarga la primera vez (~120MB) y se cachea localmente.

print(f"\n[3/5] Cargando modelo '{MODEL_NAME}'...")
model = SentenceTransformer(MODEL_NAME)
print(f"  Dimensión de embeddings: {model.get_sentence_embedding_dimension()}")

print(f"\n  Generando embeddings para {len(texts):,} textos...")
t0 = time.time()

embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,  # necesario para similitud coseno con IndexFlatIP
)

print(f"\n  Embeddings generados en {time.time()-t0:.1f}s")
print(f"  Shape: {embeddings.shape}")  # esperado: (n_lugares, 384)


In [ ]:
# ─── 4. Construir índice FAISS ────────────────────────────────────────────────
#
# IndexFlatIP: búsqueda exacta por producto interior
# Con vectores normalizados equivale a similitud coseno.
# Para ~4.600 lugares es suficiente sin índices aproximados.
#
# Score de similitud coseno:
#   1.0  = idénticos
#   0.8+ = muy similar
#   0.6+ = relacionado
#   < 0.5 = poco relevante

print("\n[4/5] Construyendo índice FAISS...")
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings.astype(np.float32))
print(f"  Vectores en el índice: {index.ntotal:,}")


In [ ]:
# ─── 5. Guardar outputs ───────────────────────────────────────────────────────

print("\n[5/5] Guardando outputs...")

index_path    = os.path.join(OUTPUT_DIR, "faiss_index.index")
metadata_path = os.path.join(OUTPUT_DIR, "faiss_metadata.json")

faiss.write_index(index, index_path)
print(f"  Índice FAISS : {index_path}  ({os.path.getsize(index_path)//1024} KB)")

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)
print(f"  Metadatos    : {metadata_path}  ({os.path.getsize(metadata_path)//1024} KB)")

print(f"\n✓ Índice construido correctamente")
print(f"  Estos dos ficheros son todo lo que necesita el backend RAG:")
print(f"    - faiss_index.index      → el índice vectorial")
print(f"    - faiss_metadata.json    → metadatos de cada lugar (mismo orden que el índice)")


## 5. Paso 2 — Verificar que el índice funciona <a name="5-verificacion"></a>

Antes de construir el backend, comprobamos que las búsquedas semánticas devuelven resultados con sentido.

### ¿Qué buscamos al verificar?

- Que los scores sean razonables (> 0.7 para consultas claras)
- Que los resultados sean semánticamente coherentes con la pregunta
- Que haya diversidad de categorías cuando la pregunta es genérica

### Interpretación de scores

| Score | Interpretación |
|-------|---------------|
| > 0.80 | Muy alta similitud |
| 0.70 - 0.80 | Buena similitud |
| 0.60 - 0.70 | Similitud moderada |
| < 0.50 | Baja similitud — resultados poco fiables |

### Limitaciones conocidas

- **Nombres propios** ("Guggenheim"): FAISS puede no encontrarlos bien si la descripción en el CSV es pobre. Solución futura: búsqueda híbrida (semántica + texto exacto).
- **Categorías poco representadas** ("actividades para niños"): si los datos no tienen esa etiqueta, el modelo no tiene de dónde tirar. Scores bajos son una señal de alerta.
- **El LLM compensa parte de estos fallos**: recuperamos k=10 candidatos y dejamos que el LLM seleccione los más relevantes.


In [ ]:
# ─── Función de búsqueda para pruebas ────────────────────────────────────────

def buscar(pregunta: str, k: int = 5):
    """
    Convierte la pregunta a embedding, busca en FAISS y muestra los k resultados
    más similares con su score de similitud coseno.
    """
    print(f"\nPregunta: '{pregunta}'")
    print("-" * 55)

    # Mismo modelo y misma normalización que al construir el índice
    embedding = model.encode([pregunta], normalize_embeddings=True)
    scores, indices = index.search(embedding.astype(np.float32), k)

    for score, idx in zip(scores[0], indices[0]):
        lugar = metadata[idx]
        rating = f" ⭐{lugar['google_rating']}" if lugar.get("google_rating") else ""
        print(f"  [{score:.3f}]{rating} {lugar['nombre']} — {lugar['municipio']} ({lugar['categoria']})")

print("✓ Función buscar() lista")


In [ ]:
# ─── Pruebas de búsqueda ─────────────────────────────────────────────────────
# Modifica estas consultas para probar lo que necesites

buscar("restaurante de pintxos en Bilbao")
buscar("playa tranquila en Gipuzkoa")
buscar("museo de arte contemporáneo")
buscar("alojamiento rural en el campo")
buscar("actividades para niños")
buscar("Guggenheim Bilbao")
buscar("museo arte moderno Bilbao")


In [ ]:
# ─── Análisis de cobertura de reseñas ────────────────────────────────────────
# Cuántos lugares tienen reseñas incorporadas y cómo se distribuyen por categoría

df_meta = pd.DataFrame(metadata)
df_meta["tiene_reviews"] = df_meta["id"].isin(reviews_by_id.keys())

print("Cobertura de reseñas por categoría:")
print()
resumen = (df_meta.groupby("categoria")
           .agg(
               total=("id", "count"),
               con_reviews=("tiene_reviews", "sum"),
           ))
resumen["pct_reviews"] = (resumen["con_reviews"] / resumen["total"] * 100).round(1)
print(resumen.sort_values("total", ascending=False).to_string())


## 6. Próximos pasos <a name="6-proximos"></a>

Con el índice FAISS construido y verificado, los siguientes pasos son:

### Paso 3 — Backend FastAPI

Un servidor que expone un endpoint `/chat` y ejecuta el pipeline completo:

```
POST /chat
{"pregunta": "restaurante de pintxos cerca del Guggenheim"}

→ {"respuesta": "Te recomiendo el Bar Txiriboga, situado a 200m del Guggenheim..."}
```

### Paso 4 — Diseño del prompt

El system prompt define el comportamiento del chatbot. Es la pieza donde más se puede afinar la calidad de las respuestas:

```
Eres Txoko, un asistente de recomendaciones turísticas de Euskadi.
Responde SIEMPRE en el idioma en que te hablen.
Usa SOLO la información de los lugares que te proporciono.
Si no tienes información suficiente, dilo claramente.
No inventes horarios, precios ni distancias.
```

### Paso 5 — Despliegue en Render

Los ficheros `faiss_index.index` y `faiss_metadata.json` se suben junto al código del backend. Render carga el índice en memoria al arrancar.

**Consideración importante con el tier gratuito de Render:** el servicio se "duerme" tras inactividad. Al despertar tarda unos segundos en cargar el índice. Gestionar esto en el frontend con un mensaje de "conectando..." evita que parezca un error.

---

### Estructura de ficheros del proyecto RAG

```
rag/
├── .env                      ← API keys (NUNCA al repositorio)
├── .gitignore                ← debe incluir .env y rag_assets/
├── build_faiss_index.py      ← este notebook convertido a script
├── aupa_master_v5.csv        ← datos fuente
├── txoko_reviews_raw.json    ← reseñas fuente
├── rag_assets/
│   ├── faiss_index.index     ← índice vectorial (generado)
│   └── faiss_metadata.json   ← metadatos (generado)
├── main.py                   ← backend FastAPI (siguiente paso)
└── requirements.txt          ← dependencias para Render
```

### requirements.txt para Render

```
fastapi
uvicorn
sentence-transformers
faiss-cpu
numpy
pandas
python-dotenv
google-generativeai
```


---

## 7. Paso 3 — Backend FastAPI: el pipeline completo

Con el índice FAISS construido, el siguiente paso es el servidor que orquesta todo el pipeline en tiempo real.

### Estructura del proyecto backend

```
backend/
├── main.py              ← servidor FastAPI, endpoint /chat
├── requirements.txt     ← dependencias para Render
├── .env                 ← API keys (nunca al repositorio)
└── rag/
    ├── config.py        ← configuración centralizada
    ├── retrieval.py     ← encode query → FAISS → filtros
    ├── prompt.py        ← construcción del prompt
    ├── llm.py           ← llamada al LLM
    └── landmarks.py     ← coordenadas de puntos de referencia
```

### El endpoint /chat

```python
@app.post("/chat")
def chat(request: ChatRequest) -> ChatResponse:
    query = request.query.strip()
    
    # 1. Buscar en FAISS
    places = retrieve(query, state.model, state.index, state.metadata)
    
    # 2. Construir prompt con los lugares recuperados
    prompt, system = build_prompt(query, places)
    
    # 3. Llamar al LLM
    answer = call_llm(prompt, system)
    
    return ChatResponse(answer=answer)
```

### Elección del LLM: de Gemini a Groq

Durante el desarrollo probamos varias opciones:

| LLM | Problema encontrado |
|-----|-------------------|
| `gemini-2.5-flash` | Cuota gratuita muy limitada, se agota rápido |
| `gemini-2.0-flash` | `limit: 0` en tier gratuito desde España |
| `gemini-1.5-flash` | Modelo no encontrado en API v1beta |
| **Groq + Llama 3** ✅ | Tier gratuito generoso, sin restricciones geográficas |

**Lección aprendida:** el tier gratuito de Gemini tiene restricciones geográficas y de modelo que no están bien documentadas. Groq es más predecible para desarrollo.

```python
# llm.py
from groq import Groq
from .config import GROQ_API_KEY, GROQ_MODEL

def call_llm(prompt: str, system_prompt: str) -> str:
    response = _client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": prompt},
        ],
        temperature=0.7,
        max_tokens=1024,
    )
    return response.choices[0].message.content
```

Una ventaja importante de Groq sobre Gemini: el `system_prompt` va en un mensaje con `role: system` separado del contenido del usuario. Esto garantiza que el LLM lo trata con prioridad.


## 8. Paso 4 — Problemas encontrados y soluciones

Esta sección documenta los bugs y limitaciones que encontramos durante el desarrollo y cómo los resolvimos. Es una parte honesta y útil del proceso.

### 8.1 Scores anómalos en FAISS (bug crítico)

**Síntoma:** los scores de similitud devueltos por FAISS eran mayores que 1 (ej: 2.67, 2.75). Con `IndexFlatIP` y vectores normalizados los scores deberían estar entre 0 y 1.

**Causa:** en la función `encode_query()` del backend faltaba `normalize_embeddings=True`. Los vectores de las queries no estaban normalizados, mientras que los del índice sí. Eso rompía la métrica de similitud coseno.

**Solución:**
```python
# MAL — faltaba normalize_embeddings
def encode_query(query, model):
    vector = model.encode([query], convert_to_numpy=True)
    return vector.astype("float32")

# BIEN
def encode_query(query, model):
    vector = model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    return vector.astype("float32")
```

**Lección:** el modelo de embeddings debe usarse exactamente igual en la fase de indexación y en la fase de consulta. Cualquier diferencia (normalización, batch_size, etc.) rompe la comparabilidad de los vectores.

---

### 8.2 El backend cargaba el índice antiguo

**Síntoma:** después de regenerar el índice los scores no mejoraban.

**Causa:** `--reload` de uvicorn recarga el código Python cuando cambia un `.py`, pero **no recarga ficheros de datos** como `.index`. El proceso seguía usando el índice antiguo cargado en memoria.

**Solución:** parar y arrancar uvicorn manualmente cada vez que se regenera el índice:
```bash
# Ctrl+C para parar
uvicorn main:app --reload
```

---

### 8.3 El LLM respondía siempre en castellano

**Síntoma:** aunque el usuario escribiera en inglés, el chatbot respondía en castellano.

**Causa raíz:** dos problemas encadenados:
1. Con Gemini: el `system_instruction` no se enviaba por el canal correcto (se mezclaba con el contenido como texto plano)
2. Con Groq/Llama: el modelo ignoraba la instrucción de idioma cuando el system prompt estaba escrito en castellano

**Solución final:** detectar el idioma de la query con `langdetect` e inyectar la instrucción de idioma directamente en el mensaje del usuario:

```python
from langdetect import detect, LangDetectException

def build_prompt(query: str, places: list[dict]) -> tuple[str, str]:
    try:
        lang = detect(query)
    except LangDetectException:
        lang = "es"
    
    if lang == "en":
        lang_instruction = "IMPORTANT: The user is writing in English. You MUST respond in English."
    elif lang == "eu":
        lang_instruction = "GARRANTZITSUA: Erabiltzaileak euskaraz idazten du. Erantzun euskaraz."
    else:
        lang_instruction = "IMPORTANTE: El usuario escribe en español. Responde en español."
    
    user_section = (
        f"{lang_instruction}\n\n"
        f'El usuario pregunta: "{query}"\n\n'
        f"Lugares disponibles:\n\n"
        f"{places_block}"
    )
    return user_section, SYSTEM_PROMPT
```

**Lección:** los LLMs pequeños (7B-8B parámetros) no siguen bien instrucciones complejas de idioma en el system prompt. Es más fiable reforzarlas en el mensaje del usuario.


## 9. Paso 5 — Mejoras de retrieval: filtros post-FAISS

FAISS devuelve los k lugares más similares semánticamente, pero eso no siempre es suficiente. Añadimos una capa de filtros post-FAISS para mejorar la relevancia geográfica y temática.

### El problema de las consultas geográficas

FAISS no entiende geografía. "Cerca del Guggenheim" no le dice nada porque ningún lugar en la BD tiene ese texto. Solo entiende similitud semántica de texto.

La solución es una cadena de filtros aplicados en orden de prioridad después de la búsqueda FAISS:

```
FAISS devuelve k=50 candidatos
        ↓
¿Menciona un landmark conocido? → reordenar por distancia haversine
        ↓
¿Menciona un municipio? → filtrar por municipio
        ↓
¿Menciona un territorio? → filtrar por Gipuzkoa/Bizkaia/Araba
        ↓
Devolver los mejores FINAL_TOP_K
```

### 9.1 Landmarks: puntos de referencia turísticos

Diccionario con ~60 puntos de referencia de Bilbao, Donostia y Vitoria-Gasteiz con sus coordenadas GPS.

```python
LANDMARKS = {
    "guggenheim":        (43.2686, -2.9340),
    "casco viejo":       (43.2587, -2.9236),
    "san mamés":         (43.2643, -2.9494),
    "parte vieja":       (43.3224, -1.9847),
    "la concha":         (43.3182, -1.9860),
    "kursaal":           (43.3233, -1.9765),
    "artium":            (42.8475, -2.6676),
    # ... ~60 landmarks en total
}
```

Cuando la query menciona un landmark, los candidatos de FAISS se reordenan por distancia haversine al punto de referencia. Los más cercanos van primero.

```python
def reorder_by_proximity(candidates, landmark_coords, max_distance_m=2000):
    # Calcula distancia haversine para cada candidato con coordenadas
    # Ordena por distancia ascendente
    # Añade _distance_m al metadata para que el LLM pueda mencionarla
    ...
```

El LLM entonces puede decir "a 350m del Guggenheim" con datos reales, no inventados.

### 9.2 Filtro por municipio

Lista de ~30 municipios principales de Euskadi. Si la query menciona uno, se filtran los candidatos:

```python
MUNICIPIOS = ["bilbao", "donostia", "san sebastián", "vitoria", "getxo", ...]

def _extract_municipio(query):
    for m in MUNICIPIOS:
        if re.search(rf"\b{re.escape(m)}\b", query.lower()):
            return m
    return None
```

### 9.3 Filtro por territorio

Detecta si la query menciona Gipuzkoa, Bizkaia o Araba y filtra los candidatos por el campo `territorio` del CSV:

```python
TERRITORIOS = {
    "gipuzkoa": "GIPUZKOA",
    "guipúzcoa": "GIPUZKOA",
    "bizkaia": "BIZKAIA",
    "vizcaya": "BIZKAIA",
    "araba": "ARABA",
    "álava": "ARABA",
}
```

**Por qué es necesario:** sin este filtro, "playa tranquila en Gipuzkoa" devolvía playas de Bizkaia porque FAISS solo entiende similitud semántica, no geografía administrativa.

### 9.4 La función apply_filters completa

```python
def apply_filters(candidates, query):
    
    # Prioridad 1: landmark (más específico)
    landmark = find_landmark(query)
    if landmark:
        _, coords = landmark
        candidates = reorder_by_proximity(candidates, coords)
        return candidates[:FINAL_TOP_K]
    
    # Prioridad 2: municipio concreto
    municipio = _extract_municipio(query)
    if municipio:
        filtered = [c for c in candidates 
                    if municipio in (c.get("municipio") or "").lower()]
        if len(filtered) >= 2:
            return filtered[:FINAL_TOP_K]
    
    # Prioridad 3: territorio
    territorio = _extract_territorio(query)
    if territorio:
        filtered = [c for c in candidates 
                    if (c.get("territorio") or "").upper() == territorio]
        if len(filtered) >= 2:
            return filtered[:FINAL_TOP_K]
    
    # Sin filtro: devolver por score semántico
    return candidates[:FINAL_TOP_K]
```


## 10. Paso 6 — El system prompt: diseño y lecciones

El system prompt es la instrucción permanente que define el comportamiento del chatbot. Es la pieza donde más impacto tiene el diseño sobre la calidad de las respuestas.

### El system prompt final

```python
SYSTEM_PROMPT = (
    "CRITICAL INSTRUCTION: Always respond in the same language the user writes in. "
    "If the user writes in English, respond in English. "
    "If the user writes in Spanish, respond in Spanish. "
    "If the user writes in Basque (Euskera), respond in Basque. "
    "Never respond in a different language than the one the user used.\n\n"
    "Eres un asistente de recomendaciones turísticas de Euskadi llamado Aupa.\n"
    "Usa únicamente la información proporcionada en el contexto.\n"
    "No inventes datos como horarios, precios ni distancias.\n"
    "Si no tienes suficiente información, dilo claramente.\n"
    "Si los lugares no encajan bien con la pregunta, dilo explícitamente.\n"
    "Si la pregunta es geográfica como 'cerca de X', advierte que no puedes "
    "garantizar proximidad exacta y sugiere verificar en el mapa.\n"
    "Presenta máximo 3 recomendaciones, no todas las disponibles.\n"
    "Sé breve y directo, máximo 3-4 frases por recomendación.\n"
    "Si el usuario pide algo auténtico, local o alejado del turismo masivo, "
    "prioriza los lugares con mayor puntuación de autenticidad local. "
    "Una puntuación de 60/100 o más indica un lugar genuinamente local.\n"
)
```

### Decisiones de diseño del prompt

**¿Por qué la instrucción de idioma va en inglés al principio?**
Los LLMs dan más peso a las instrucciones al inicio del prompt. Escribirla en inglés ayuda porque los modelos tienen más datos de entrenamiento en inglés y entienden mejor las instrucciones en ese idioma, aunque el contenido sea en español.

**¿Por qué "máximo 3 recomendaciones"?**
Sin esta restricción el LLM tiende a listar todos los lugares disponibles en el contexto, lo que produce respuestas largas y poco útiles para el usuario.

**¿Por qué "no inventes horarios ni precios"?**
Las alucinaciones son el riesgo principal en RAG. El LLM puede completar información que no tiene con datos plausibles pero incorrectos. Esta instrucción reduce significativamente ese problema.

**La instrucción de autenticidad local**
El campo `local_ratio` del CSV (valor entre 0 y 1, escalado a 100) mide qué tan frecuentado es un lugar por locales vs turistas. Se incluye en el contexto de cada lugar y el LLM lo usa cuando el usuario pide algo "auténtico" o "sin turistas".

### Lecciones sobre prompt engineering

| Problema | Causa | Solución |
|----------|-------|----------|
| Responde en idioma equivocado | Instrucción de idioma débil o en el idioma equivocado | Instrucción en inglés + detección con langdetect |
| Inventa horarios o precios | LLM completa información que no tiene | Instrucción explícita de no inventar |
| Lista todos los lugares | Sin restricción de cantidad | "Máximo 3 recomendaciones" |
| Recomienda lugares irrelevantes | Scores bajos de FAISS sin umbral | Umbral mínimo de score + más candidatos en FAISS |


## 11. Autenticidad local: el campo local_ratio

Una característica diferencial de Txoko es que prioriza recomendaciones auténticas y locales sobre las turísticas. El campo `local_ratio` del CSV cuantifica esto.

### Qué mide local_ratio

Es un score entre 0 y 1 calculado a partir de señales como la distribución de idiomas en las reseñas, el tipo de categoría, y otros indicadores. Se escala a 100 para el usuario.

| Score | Interpretación |
|-------|---------------|
| > 60 | Lugar genuinamente local, frecuentado por residentes |
| 40-60 | Mix de turistas y locales |
| < 40 | Principalmente turístico |

**Nota importante:** los valores máximos en el dataset son ~70/100. Un 65/100 no significa que sea mediocre, significa que es de los lugares más locales del dataset.

### Cómo se incorpora al pipeline

**En el índice:** `local_ratio` está en los metadatos de cada lugar (`faiss_metadata.json`).

**En el prompt:** se formatea como "Autenticidad local: 62/100" en el bloque de cada lugar.

**En el system prompt:** el LLM tiene instrucciones para priorizar lugares con score ≥ 60 cuando el usuario pide algo auténtico.

```python
# En _format_place() de prompt.py
local_ratio = place.get("local_ratio")
if local_ratio is not None:
    local_label = f"{int(float(local_ratio) * 100)}/100"
    lines.append(f"   Autenticidad local: {local_label}")
```


## 12. Limitaciones conocidas y trabajo futuro

### Limitaciones actuales (honestas para el jurado)

| Limitación | Causa | Posible solución futura |
|------------|-------|------------------------|
| No sabe horarios ni precios | Open Data Euskadi no los publica | Integrar API de Google Places Details |
| Consultas muy vagas dan resultados dispares | FAISS necesita contexto semántico | Mejorar descripciones en el CSV |
| "Actividades para niños" funciona mal | Categoría no etiquetada en los datos | Añadir etiquetas manualmente |
| Nombres propios poco conocidos fallan | No están en los textos embeddados | Búsqueda híbrida: semántica + texto exacto |
| Eventos en tiempo real no disponibles | No están en Open Data Euskadi | Integrar API de agenda de Bilbao/Eventbrite |
| Cobertura geográfica limitada a Euskadi | Solo tenemos datos de Euskadi | Ampliar con otras fuentes |

### Qué funciona bien

- Consultas con categoría + ubicación: "restaurante en Bilbao", "museo en Vitoria"
- Consultas con landmark: "cerca del Guggenheim", "junto a la Parte Vieja"
- Consultas de autenticidad: "algo local sin turistas"
- Multilingüe: español, inglés, euskera
- Distancias reales a puntos de referencia

### Mejoras técnicas pendientes

**Búsqueda híbrida (BM25 + embeddings):** combinar búsqueda por texto exacto con búsqueda semántica. Mejoraría resultados para nombres propios y consultas específicas.

**Re-ranking:** usar un modelo de re-ranking (cross-encoder) para reordenar los candidatos de FAISS con más precisión antes de pasarlos al LLM.

**Caché de respuestas:** guardar respuestas a consultas frecuentes para reducir latencia y consumo de API.

**Evaluación sistemática:** construir un conjunto de 50-100 preguntas con respuesta esperada y medir la calidad del sistema automáticamente.

---

## Resumen del pipeline completo

```
[OFFLINE — ejecutar una vez]
aupa_master_v7.csv + txoko_reviews_raw.json
        ↓
build_faiss_index.py
        ↓
faiss_index.index + faiss_metadata.json

[ONLINE — cada consulta]
Usuario: "pintxos cerca del Guggenheim"
        ↓
langdetect → idioma = "es"
        ↓
encode_query() → embedding normalizado (384 dims)
        ↓
FAISS search k=50 → candidatos por similitud semántica
        ↓
apply_filters():
  → find_landmark("guggenheim") → reorder_by_proximity()
        ↓
build_prompt():
  → lang_instruction + lugares formateados + local_ratio + distancias
        ↓
call_llm() → Groq + Llama 3.3 70B
  → system_prompt (instrucciones fijas)
  → user prompt (contexto dinámico)
        ↓
Respuesta en lenguaje natural con distancias reales
```
